# **Gini Coefficient Measures — Global Income Inequality Analysis & Forecasting**

**Objective:** clean a country-year Gini coefficient dataset (1963–2025), explore global inequality trends,
and build a machine-learning model that predicts an entity's Gini coefficient from its recent history.

**Workflow:** data inspection → cleaning (fixing a locale-corrupted numeric column) → EDA → feature engineering →
time-based train/test split → Linear Regression vs. Random Forest vs. tuned Random Forest (time-series CV) →
evaluation against a naive baseline → model export and prediction function.

## **Import**

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [ ]:
# The Gini column must be read as text: its values contain several dots (e.g. '3.672.647.476.196.280')
# and are repaired in the cleaning section.
candidates = [
    "Gini-Coefficient-Measures_new.csv",
    "Gini-Coefficient-Measures new.csv",
    "/content/Gini-Coefficient-Measures new.csv",
    "/content/Gini-Coefficient-Measures_new.csv",
    "/mnt/user-data/uploads/Gini-Coefficient-Measures_new.csv",
]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place Gini-Coefficient-Measures_new.csv next to this notebook.")

df = pd.read_csv(DATA_PATH, dtype={"Gini coefficient": str})
display(df.head())

In [ ]:
display(df.tail())

In [ ]:
# Dataset shape
print("Dataset Shape:")
display(df.shape)

# Column names
print("\nColumn Names:")
display(df.columns.tolist())

# First 5 rows
print("\nFirst 5 Rows:")
display(df.head())

# Last 5 rows
print("\nLast 5 Rows:")
display(df.tail())

# Data types
print("\nData Types:")
display(df.dtypes)

# Dataset information
print("\nDataset Information:")
df.info()

# Missing values
print("\nMissing Values:")
display(df.isnull().sum())

# Unique values
print("\nUnique Values:")
display(df.nunique())

In [ ]:
print("Column Names:")
display(df.columns.tolist())

## **Detailed Dataset Inspection**

**1. Check duplicate rows**

In [ ]:
# Count duplicate rows
df.duplicated().sum()

**2. Check the year range**

In [ ]:
print("Minimum Year:", df["Year"].min())
print("Maximum Year:", df["Year"].max())

In [ ]:
sorted(df["Year"].unique())

**3. Check number of countries/entities**

In [ ]:
print("Number of Entities:", df["Entity"].nunique())

In [ ]:
df["Entity"].unique()

In [ ]:
for entity in sorted(df["Entity"].dropna().unique()):
    display(entity)

**4. Check number of country codes**

In [ ]:
print("Number of Codes:", df["Code"].nunique())

In [ ]:
df["Code"].unique()

**5. Find entities without a code**

In [ ]:
df[df["Code"].isna()]["Entity"].unique()

In [ ]:
df["Code"].isna().sum()

**6. Check observations per entity**

In [ ]:
entity_counts = df["Entity"].value_counts()

display(entity_counts)

In [ ]:
entity_counts.head(20)

In [ ]:
entity_counts.tail(20)

**7. Check observations per year**

In [ ]:
year_counts = df["Year"].value_counts().sort_index()

print(year_counts)

In [ ]:
plt.figure(figsize=(12, 5))

year_counts.plot(kind="bar")

plt.title("Number of Observations by Year")
plt.xlabel("Year")
plt.ylabel("Number of Observations")
plt.xticks(rotation=90)
plt.show()

**8. Check duplicate Entity-Year combinations**

In [ ]:
duplicate_entity_year = df[
    df.duplicated(subset=["Entity", "Year"], keep=False)
]

duplicate_entity_year

In [ ]:
df.duplicated(subset=["Entity", "Year"]).sum()

**9. Inspect the Gini coefficient column**

In [ ]:
df["Gini coefficient"].head(20)

In [ ]:
df["Gini coefficient"].unique()[:20]

In [ ]:
type(df["Gini coefficient"].iloc[0])

**10. Inspect GC Percentage**

In [ ]:
df["GC Percentage"].head(20)

In [ ]:
df["GC Percentage"].unique()[:20]

In [ ]:
type(df["GC Percentage"].iloc[0])

**11. Statistical summary**

In [ ]:
df.describe()

In [ ]:
df.describe(include="all")

## **Data Cleaning & Preprocessing**

> **Data scope notes**
> - `Code == "-"` marks **sub-national series** (e.g. *China (rural)*, *Argentina (urban)*); they are flagged in `Is_Subnational`.
> - Coverage is uneven: not every entity is observed every year, so a "lag" means the *previous available observation*.
> - The raw `GC Percentage` column is corrupted and is replaced by `Gini × 100` (shown in the validation section).

**1. Make a copy first**

In [ ]:
df_clean = df.copy()

**2. Remove unnecessary spaces from column names**

In [ ]:
df_clean.columns = df_clean.columns.str.strip()

**3. Clean text columns**

In [ ]:
df_clean["Entity"] = df_clean["Entity"].str.strip()
df_clean["Code"] = df_clean["Code"].str.strip()

**4. Convert Gini coefficient to numeric**

In [ ]:
df_clean["Gini coefficient"].head(10)

In [ ]:
# ROOT CAUSE: the export wrote the decimal digits of a 0-1 Gini value with '.' used as a thousands separator,
# e.g. '3.672.647.476.196.280' is really 0.3672647476196280 (trailing zeros are sometimes dropped,
# e.g. '287.769.079.208.374' = 0.287769079208374).
# Correct rule: keep only the digits and read them as the fractional part -> 0.<digits>.
def clean_gini_string(val):
    if pd.isna(val):
        return np.nan
    digits = re.sub(r"\D", "", str(val))
    return float("0." + digits) if digits else np.nan

df_clean["Gini coefficient"] = df["Gini coefficient"].apply(clean_gini_string)

# Flag sub-national series (urban/rural) which have no country code ('-')
df_clean["Is_Subnational"] = df_clean["Code"].eq("-")

In [ ]:
print("Missing Gini values:", df_clean["Gini coefficient"].isna().sum())
print("Sub-national rows (urban/rural series):", int(df_clean["Is_Subnational"].sum()))

**5. Clean GC Percentage (raw column is corrupted — see validation below)**

In [ ]:
df_clean["GC Percentage"] = (
    df_clean["GC Percentage"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False)
)

In [ ]:
df_clean["GC Percentage"] = pd.to_numeric(
    df_clean["GC Percentage"],
    errors="coerce"
)

In [ ]:
df_clean["GC Percentage"].head()

**6. Check data types again**

In [ ]:
df_clean.dtypes

**7. Check missing values after conversion**

In [ ]:
df_clean.isnull().sum()

**8. Check duplicates**

In [ ]:
print("Duplicate rows:", df_clean.duplicated().sum())

In [ ]:
df_clean = df_clean.drop_duplicates()

**9. Check Entity-Year duplicates**

In [ ]:
df_clean.duplicated(
    subset=["Entity", "Year"]
).sum()

In [ ]:
df_clean[
    df_clean.duplicated(
        subset=["Entity", "Year"],
        keep=False
    )
].sort_values(["Entity", "Year"])

**10. Check numerical ranges**

In [ ]:
print(df_clean["Gini coefficient"].describe())
print(df_clean["GC Percentage"].describe())

In [ ]:
print("Minimum Gini:", df_clean["Gini coefficient"].min())
print("Maximum Gini:", df_clean["Gini coefficient"].max())

## **Fix & Validate the Gini Coefficient Values**

**1. Inspect the original values**

In [ ]:
df["Gini coefficient"].head(20).tolist()

In [ ]:
print(df["Gini coefficient"].dtype)

**2. Inspect GC Percentage**

In [ ]:
df["GC Percentage"].head(20).tolist()

**3. Convert GC Percentage**

In [ ]:
# The raw 'GC Percentage' column is NOT a reliable percentage of the Gini: it is inconsistent
# (e.g. US 1963 -> 0.19% but US 1965 -> 1.90% for almost identical Gini values), so it is kept only
# for diagnostics. A correct percentage version is recomputed from the repaired Gini coefficient.
df_clean["GC Percentage Raw"] = pd.to_numeric(
    df["GC Percentage"].astype(str).str.replace("%", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)
df_clean["GC Percentage"] = df_clean["Gini coefficient"] * 100   # Gini on a 0-100 scale

display(df_clean[["Entity", "Year", "Gini coefficient", "GC Percentage", "GC Percentage Raw"]].head(10))

**4. Check the relationship between the repaired Gini and the raw percentage column**

In [ ]:
df_clean[["Gini coefficient", "GC Percentage", "GC Percentage Raw"]].head(20)

In [ ]:
df_clean[["Gini coefficient", "GC Percentage", "GC Percentage Raw"]].corr()

**5. Look for impossible Gini values (valid range is 0 to 1)**

In [ ]:
display({
    "Minimum": df_clean["Gini coefficient"].min(),
    "Maximum": df_clean["Gini coefficient"].max(),
    "Values outside [0, 1]": int((~df_clean["Gini coefficient"].between(0, 1)).sum())
})

**6. Check suspicious values**

In [ ]:
df_clean.sort_values(
    "Gini coefficient",
    ascending=False
).head(20)

In [ ]:
df_clean.sort_values(
    "Gini coefficient",
    ascending=True
).head(20)

**7. Quantify how badly the raw percentage column disagrees with the repaired Gini**

In [ ]:
df_clean["Difference"] = df_clean["GC Percentage Raw"] - df_clean["GC Percentage"]

In [ ]:
df_clean["Difference"].describe()

In [ ]:
df_clean[
    df_clean["Difference"].abs() > 1
][[
    "Entity", "Year", "Gini coefficient", "GC Percentage", "GC Percentage Raw", "Difference"
]].head(20)

## **Diagnostic: raw vs. cleaned values**

In [ ]:
display(pd.DataFrame({
    "Raw Gini (text)": df["Gini coefficient"].head(10),
    "Cleaned Gini": df_clean["Gini coefficient"].head(10),
    "Raw GC Percentage": df["GC Percentage"].head(10),
}))
print("\nCleaned Gini data type:", df_clean["Gini coefficient"].dtype)

## **Exploratory Data Analysis (EDA)**

**1. Dataset overview**

In [ ]:
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

display(df_clean.head())
display(df_clean.describe())

**Gini Coefficient Over Time**

In [ ]:
yearly_gini = (
    df_clean
    .groupby("Year", as_index=False)["Gini coefficient"]
    .mean()
)

fig = px.line(
    yearly_gini,
    x="Year",
    y="Gini coefficient",
    markers=True,
    title="Average Gini Coefficient Over Time",
    labels={
        "Year": "Year",
        "Gini coefficient": "Average Gini Coefficient"
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

**Gini by Country**

In [ ]:
country_gini = (
    df_clean
    .groupby("Entity", as_index=False)["Gini coefficient"]
    .mean()
    .sort_values("Gini coefficient", ascending=False)
)

top_15 = country_gini.head(15)

fig = px.bar(
    top_15,
    x="Gini coefficient",
    y="Entity",
    orientation="h",
    title="Top 15 Entities by Average Gini Coefficient",
    labels={
        "Gini coefficient": "Average Gini Coefficient",
        "Entity": "Entity"
    }
)

fig.update_layout(
    template="plotly_white",
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

**Distribution of Gini Coefficient**

In [ ]:
fig = px.histogram(
    df_clean,
    x="Gini coefficient",
    nbins=30,
    title="Distribution of Gini Coefficient",
    labels={
        "Gini coefficient": "Gini Coefficient"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Box Plot**

In [ ]:
fig = px.box(
    df_clean,
    y="Gini coefficient",
    title="Distribution and Outliers of Gini Coefficient",
    labels={
        "Gini coefficient": "Gini Coefficient"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Country vs Year Heatmap**

In [ ]:
heatmap_data = df_clean.pivot_table(
    index="Entity",
    columns="Year",
    values="Gini coefficient",
    aggfunc="mean"
)

fig = px.imshow(
    heatmap_data,
    aspect="auto",
    title="Gini Coefficient Heatmap by Entity and Year",
    labels={
        "x": "Year",
        "y": "Entity",
        "color": "Gini Coefficient"
    }
)

fig.show()

**Gini vs GC Percentage**

In [ ]:
fig = px.scatter(
    df_clean,
    x="Gini coefficient",
    y="GC Percentage",
    title="Gini Coefficient vs GC Percentage",
    trendline="ols",
    labels={
        "Gini coefficient": "Gini Coefficient",
        "GC Percentage": "GC Percentage"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Interactive Country-Year Analysis**

In [ ]:
fig = px.scatter(
    df_clean,
    x="Year",
    y="Gini coefficient",
    color="Entity",
    hover_data=["Code", "GC Percentage"],
    title="Interactive Gini Coefficient Over Time",
    labels={
        "Year": "Year",
        "Gini coefficient": "Gini Coefficient",
        "Entity": "Entity"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Average Gini by Year**

In [ ]:
year_summary = (
    df_clean
    .groupby("Year")
    .agg(
        Average_Gini=("Gini coefficient", "mean"),
        Minimum_Gini=("Gini coefficient", "min"),
        Maximum_Gini=("Gini coefficient", "max")
    )
    .reset_index()
)

display(year_summary.head())

In [ ]:
fig = px.line(
    year_summary,
    x="Year",
    y="Average_Gini",
    markers=True,
    title="Average Gini Coefficient by Year"
)

fig.show()

## **Feature Engineering**

**1. Create a working dataset**

In [ ]:
# 1. Create a working dataset from the newly cleaned df_clean
ml_df = df_clean.copy()

ml_df = ml_df.sort_values(
    ["Entity", "Year"]
).reset_index(drop=True)

# 2. Re-create time and decade features
ml_df["Years_Since_Start"] = ml_df["Year"] - ml_df["Year"].min()
ml_df["Decade"] = (ml_df["Year"] // 10) * 10

# 3. Create lag features
ml_df["Gini_Lag_1"] = ml_df.groupby("Entity")["Gini coefficient"].shift(1)
ml_df["Gini_Lag_2"] = ml_df.groupby("Entity")["Gini coefficient"].shift(2)
ml_df["Gini_Lag_3"] = ml_df.groupby("Entity")["Gini coefficient"].shift(3)

# 4. Create rolling averages
ml_df["Gini_Rolling_3"] = ml_df.groupby("Entity")["Gini coefficient"].transform(lambda x: x.shift(1).rolling(3).mean())
ml_df["Gini_Rolling_5"] = ml_df.groupby("Entity")["Gini coefficient"].transform(lambda x: x.shift(1).rolling(5).mean())

# 5. Create difference metrics
ml_df["Gini_Change"] = ml_df.groupby("Entity")["Gini coefficient"].diff()
ml_df["Gini_Pct_Change"] = ml_df.groupby("Entity")["Gini coefficient"].pct_change() * 100
ml_df["Entity_Avg_Gini"] = ml_df.groupby("Entity")["Gini coefficient"].transform("mean")

# Let's see if we have valid non-empty values now!
print("Missing values in ml_df:\n", ml_df[["Gini coefficient", "Gini_Lag_1", "Gini_Rolling_3"]].isnull().sum())
display(ml_df.dropna(subset=["Gini_Lag_1", "Gini_Lag_2", "Gini_Lag_3", "Gini_Rolling_3", "Gini_Rolling_5"]).head())

In [ ]:
ml_df.head()

**7.2 Create a time feature**

In [ ]:
ml_df["Years_Since_Start"] = (
    ml_df["Year"] - ml_df["Year"].min()
)

In [ ]:
ml_df[["Entity", "Year", "Years_Since_Start"]].head()

**3. Create decade feature**

In [ ]:
ml_df["Decade"] = (
    ml_df["Year"] // 10
) * 10

In [ ]:
ml_df[["Year", "Decade"]].head(20)

**4. Create lag features**

**Previous year**

In [ ]:
ml_df["Gini_Lag_1"] = (
    ml_df
    .groupby("Entity")["Gini coefficient"]
    .shift(1)
)

**Two years earlier**

In [ ]:
ml_df["Gini_Lag_2"] = (
    ml_df
    .groupby("Entity")["Gini coefficient"]
    .shift(2)
)

**Three years earlier**

In [ ]:
ml_df["Gini_Lag_3"] = (
    ml_df
    .groupby("Entity")["Gini coefficient"]
    .shift(3)
)

In [ ]:
ml_df[
    [
        "Entity",
        "Year",
        "Gini coefficient",
        "Gini_Lag_1",
        "Gini_Lag_2",
        "Gini_Lag_3"
    ]
].head(20)

**5. Create rolling averages**

**3-year rolling average**

In [ ]:
ml_df["Gini_Rolling_3"] = (
    ml_df
    .groupby("Entity")["Gini coefficient"]
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

**5-year rolling average**

In [ ]:
ml_df["Gini_Rolling_5"] = (
    ml_df
    .groupby("Entity")["Gini coefficient"]
    .transform(
        lambda x: x.shift(1).rolling(5).mean()
    )
)

**6. Create year-over-year change**

In [ ]:
ml_df["Gini_Change"] = (
    ml_df
    .groupby("Entity")["Gini coefficient"]
    .diff()
)

**7. Create percentage change**

In [ ]:
ml_df["Gini_Pct_Change"] = (
    ml_df
    .groupby("Entity")["Gini coefficient"]
    .pct_change()
) * 100

In [ ]:
ml_df[
    [
        "Entity",
        "Year",
        "Gini coefficient",
        "Gini_Change",
        "Gini_Pct_Change"
    ]
].head(20)

**8. Create an entity-level average**

In [ ]:
entity_mean = (
    ml_df
    .groupby("Entity")["Gini coefficient"]
    .transform("mean")
)

ml_df["Entity_Avg_Gini"] = entity_mean

**9. Create an entity trend feature**

In [ ]:
def calculate_trend(group):
    if len(group) < 2:
        return np.nan

    return np.polyfit(
        group["Year"],
        group["Gini coefficient"],
        1
    )[0]

entity_trend = (
    ml_df
    .groupby("Entity")
    .apply(calculate_trend)
)

ml_df["Entity_Gini_Trend"] = (
    ml_df["Entity"]
    .map(entity_trend)
)

**10. Check all features**

In [ ]:
ml_df.columns

**11. Visualize the new features with Plotly**

In [ ]:
feature_df = ml_df.dropna(
    subset=["Gini_Lag_1", "Gini coefficient"]
)

fig = px.scatter(
    feature_df,
    x="Gini_Lag_1",
    y="Gini coefficient",
    color="Entity",
    hover_name="Entity",
    hover_data=["Year"],
    title="Previous-Year Gini vs Current Gini",
    labels={
        "Gini_Lag_1": "Previous-Year Gini",
        "Gini coefficient": "Current Gini"
    }
)

fig.update_layout(template="plotly_white")

fig.show()

**12. Check missing values created by feature engineering**

In [ ]:
ml_df.isnull().sum()

**13. Prepare the Machine Learning dataset (chronologically ordered)**

In [ ]:
model_df = ml_df.dropna(
    subset=["Gini_Lag_1", "Gini_Lag_2", "Gini_Lag_3", "Gini_Rolling_3", "Gini_Rolling_5"]
).sort_values(["Year", "Entity"]).reset_index(drop=True).copy()

print("Original rows:", len(ml_df))
print("ML rows ready for training:", len(model_df))

In [ ]:
print("Original rows:", len(ml_df))
print("ML rows:", len(model_df))

## **Train/Test Split + Machine Learning**

**1. Select ML features**

In [ ]:
features = [
    "Year",
    "Years_Since_Start",
    "Gini_Lag_1",
    "Gini_Lag_2",
    "Gini_Lag_3",
    "Gini_Rolling_3",
    "Gini_Rolling_5"
]

target = "Gini coefficient"

In [ ]:
X = model_df[features]
y = model_df[target]

In [ ]:
shape_summary = pd.DataFrame({
    "Dataset": ["X (Features)", "y (Target)"],
    "Shape": [X.shape, y.shape]
})

print("===== DATASET SHAPE SUMMARY =====")
display(shape_summary)

**2. Time-based Train/Test Split**

In [ ]:
# 7. Time-based Train/Test Split
split_year = model_df["Year"].quantile(0.80)

train_df = model_df[model_df["Year"] <= split_year].copy()
test_df = model_df[model_df["Year"] > split_year].copy()

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

# Create split summary
split_summary = pd.DataFrame({
    "Dataset": [
        "Training Features",
        "Testing Features",
        "Training Target",
        "Testing Target"
    ],
    "Shape": [
        X_train.shape,
        X_test.shape,
        y_train.shape,
        y_test.shape
    ]
})

print("===== TIME-BASED TRAIN / TEST SPLIT =====")
display(split_summary)

print(f"\nSplit Year (80th Percentile): {split_year:.2f}")

**3. Create X_train, X_test, y_train, y_test**

In [ ]:
X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

In [ ]:
split_summary = pd.DataFrame({
    "Dataset": [
        "X_train",
        "X_test",
        "y_train",
        "y_test"
    ],
    "Shape": [
        X_train.shape,
        X_test.shape,
        y_train.shape,
        y_test.shape
    ]
})

print("===== TRAIN / TEST SPLIT SUMMARY =====")
display(split_summary)

**4. Feature Scaling**

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Scale the features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create summary table
scaling_summary = pd.DataFrame({
    "Dataset": [
        "Training Features",
        "Testing Features"
    ],
    "Status": [
        "✓ Scaled Successfully",
        "✓ Scaled Successfully"
    ],
    "Shape": [
        X_train_scaled.shape,
        X_test_scaled.shape
    ]
})

print("===== FEATURE SCALING SUMMARY =====")
display(scaling_summary)

**5. Train the Machine Learning Model**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize Random Forest Regression model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train the model
model.fit(X_train_scaled, y_train)

# Predict and evaluate on the test set
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Create results table
rf_results = pd.DataFrame({
    "Metric": [
        "Mean Squared Error (MSE)",
        "R² Score"
    ],
    "Value": [
        mse,
        r2
    ]
})

rf_results["Value"] = rf_results["Value"].round(4)

print("===== RANDOM FOREST REGRESSION RESULTS =====")
display(rf_results)

**6. Feature Importance Analysis**

In [ ]:
# Get feature importance
feature_importance = model.feature_importances_

# Create a DataFrame
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importance
})

# Sort by importance
importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

print("===== FEATURE IMPORTANCE =====")

# Round values for clean display
display(importance_df.round(4))

**7. Visualize Feature Importance**

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    importance_df['Feature'],
    importance_df['Importance']
)

plt.xlabel("Importance")
plt.ylabel("Features")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()

plt.show()

## **Compare Random Forest vs Linear Regression**

**1. Train Linear Regression**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize Linear Regression
linear_model = LinearRegression()

# Train the model
linear_model.fit(X_train_scaled, y_train)

# Predict
y_pred_linear = linear_model.predict(X_test_scaled)

# Evaluate
mse_linear = mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)

# Create results table
linear_results = pd.DataFrame({
    "Metric": [
        "Mean Squared Error (MSE)",
        "R² Score"
    ],
    "Value": [
        mse_linear,
        r2_linear
    ]
})

linear_results["Value"] = linear_results["Value"].round(4)

print("===== LINEAR REGRESSION RESULTS =====")
display(linear_results)

**2. Compare Both Models**

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MSE': [mse_linear, mse],
    'R2 Score': [r2_linear, r2]
})

print("===== MODEL COMPARISON =====")
display(comparison.round(4))

**3. Visualize R² Scores**

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(comparison['Model'], comparison['R2 Score'])

plt.ylabel("R² Score")
plt.title("Model Performance Comparison")

plt.show()

**4. Naive baseline — "next value = last observed value"**

Because inequality changes slowly, any useful model must beat simply repeating the previous observation.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred_naive = X_test["Gini_Lag_1"].values
mse_naive = mean_squared_error(y_test, y_pred_naive)
rmse_naive = np.sqrt(mse_naive)
r2_naive = r2_score(y_test, y_pred_naive)

display(pd.DataFrame({
    "Model": ["Naive baseline (last value)", "Linear Regression", "Random Forest"],
    "MSE": [mse_naive, mse_linear, mse],
    "R2 Score": [r2_naive, r2_linear, r2]
}).round(5))

## **Actual vs Predicted Values**

**1. Create Actual vs Predicted DataFrame**

In [ ]:
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

display(results.head(10))

**2. Plot Actual vs Predicted**

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(results['Actual'], results['Predicted'])

# Perfect prediction line
plt.plot(
    [results['Actual'].min(), results['Actual'].max()],
    [results['Actual'].min(), results['Actual'].max()],
    linestyle='--'
)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted — Random Forest")

plt.show()

**3. Check Prediction Errors**

In [ ]:
# Calculate prediction error
results['Error'] = results['Actual'] - results['Predicted']

print("===== ACTUAL vs PREDICTED =====")
display(results.head(10))

# Calculate Mean Absolute Error
mae_results = results['Error'].abs().mean()

print("\n===== MEAN ABSOLUTE ERROR =====")

display(pd.DataFrame({
    "Metric": ["Mean Absolute Error (MAE)"],
    "Value": [round(mae_results, 4)]
}))

## **Hyperparameter Tuning with GridSearchCV**

**1. Import GridSearchCV**

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor

**2. Define the Parameter Grid**

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

display("Parameter grid created successfully!")

**3. Run Grid Search (time-series cross-validation — no future data leaks into training folds)**

In [ ]:
rf = RandomForestRegressor(random_state=42)
tscv = TimeSeriesSplit(n_splits=5)   # train_df is already sorted by Year

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=tscv,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("===== HYPERPARAMETER TUNING =====")
display(pd.DataFrame({
    "Item": ["Algorithm", "Cross-Validation", "Scoring Metric", "Status"],
    "Details": ["Random Forest Regressor", "5-fold TimeSeriesSplit", "R² Score", "✓ Completed Successfully"]
}))

**4. Find the Best Parameters**

In [ ]:
best_parameters = pd.DataFrame({
    "Parameter": list(grid_search.best_params_.keys()),
    "Best Value": list(grid_search.best_params_.values())
})

cv_score = pd.DataFrame({
    "Metric": ["Best Cross-Validation R² Score"],
    "Score": [grid_search.best_score_]
})

cv_score["Score"] = cv_score["Score"].round(4)

print("===== BEST HYPERPARAMETERS =====")
display(best_parameters)

print("===== BEST CROSS-VALIDATION SCORE =====")
display(cv_score)

**5. Evaluate the Tuned Model**

In [ ]:
best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test_scaled)

mse_tuned = mean_squared_error(y_test, y_pred_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

# Create results table
tuned_results = pd.DataFrame({
    "Metric": [
        "Mean Squared Error (MSE)",
        "R² Score"
    ],
    "Value": [
        mse_tuned,
        r2_tuned
    ]
})

tuned_results["Value"] = tuned_results["Value"].round(4)

print("===== TUNED RANDOM FOREST RESULTS =====")
display(tuned_results)

## **Final Model Evaluation & Saving**

**1. Final Evaluation**

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Final predictions
y_pred_final = best_model.predict(X_test_scaled)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred_final)
mse_final = mean_squared_error(y_test, y_pred_final)
rmse = np.sqrt(mse_final)
r2_final = r2_score(y_test, y_pred_final)

# Create performance table
final_performance = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R² Score"],
    "Value": [mae, mse_final, rmse, r2_final]
})

final_performance["Value"] = final_performance["Value"].round(4)

print("===== FINAL MODEL PERFORMANCE =====")
display(final_performance)

**2. Create Final Prediction Table**

In [ ]:
final_results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred_final
})

final_results['Error'] = (
    final_results['Actual'] - final_results['Predicted']
)

display(final_results.head(10))

**3. Save the Model**

In [ ]:
import joblib

# Save trained model
joblib.dump(best_model, 'final_random_forest_model.pkl')

# Save scaler
joblib.dump(scaler, 'feature_scaler.pkl')

print("Model saved successfully!")
print("Scaler saved successfully!")

**4. Verify the Saved Files**

In [ ]:
import os

print("Model file exists:",
      os.path.exists('final_random_forest_model.pkl'))

print("Scaler file exists:",
      os.path.exists('feature_scaler.pkl'))

## **Build a Prediction System**

**1. Load the Model and Scaler**

In [ ]:
import joblib

# Load the trained model
loaded_model = joblib.load('final_random_forest_model.pkl')

# Load the scaler
loaded_scaler = joblib.load('feature_scaler.pkl')

print("===== MODEL LOADING STATUS =====")

display(pd.DataFrame({
    "Component": [
        "Random Forest Model",
        "Feature Scaler"
    ],
    "Status": [
        "✓ Loaded Successfully",
        "✓ Loaded Successfully"
    ],
    "File": [
        "final_random_forest_model.pkl",
        "feature_scaler.pkl"
    ]
}))

**2. Check Your Feature Names**

In [ ]:
print("===== FEATURES REQUIRED BY THE MODEL =====")

display(pd.DataFrame({
    "Feature": X_train.columns.tolist()
}))

**3. Create New Input**

In [ ]:

new_data = pd.DataFrame({
    feature: [X_train[feature].mean()]
    for feature in X_train.columns
})

print("===== NEW INPUT DATA =====")
display(new_data)

**4. Scale the New Data**

In [ ]:
new_data_scaled = loaded_scaler.transform(new_data)

print("New data scaled successfully!")

print("\n===== SCALED NEW DATA =====")
display(pd.DataFrame(
    new_data_scaled,
    columns=new_data.columns
))

**5. Make the Prediction**

In [ ]:
prediction = loaded_model.predict(new_data_scaled)

print("===== PREDICTION RESULT =====")
display({
    "Predicted Value": round(float(prediction[0]), 4)
})

**6. Complete Prediction Function**

In [ ]:
def predict_target(new_data):
    new_data_scaled = loaded_scaler.transform(new_data)
    prediction = loaded_model.predict(new_data_scaled)
    return prediction[0]

result = predict_target(new_data)

print("\n===== FINAL PREDICTION =====")
display(new_data)
print(f"Final Predicted Target: {result:.4f}")

## **ML Project Visualizations**

**1. Actual vs Predicted Values**

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_pred_final)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle='--'
)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted Values")

plt.show()

**2. Prediction Error Distribution**

In [ ]:
errors = y_test - y_pred_final

plt.figure(figsize=(8, 6))

plt.hist(errors, bins=30)

plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("Prediction Error Distribution")

plt.show()

**3. Final Feature Importance**

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

plt.figure(figsize=(10, 6))

plt.barh(
    feature_importance['Feature'],
    feature_importance['Importance']
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")

plt.gca().invert_yaxis()

plt.show()

**4. Model Performance Comparison**

In [ ]:
comparison_final = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'Tuned Random Forest'
    ],
    'R2 Score': [
        r2_linear,
        r2,
        r2_final
    ]
})

display(comparison_final)

In [ ]:
plt.figure(figsize=(9, 6))

plt.bar(
    comparison_final['Model'],
    comparison_final['R2 Score']
)

plt.xlabel("Model")
plt.ylabel("R² Score")
plt.title("Model Performance Comparison")

plt.xticks(rotation=15)

plt.show()

## **Export Final ML Results**

**1. Save Predictions**

In [ ]:
# Save actual vs predicted values
final_results.to_csv(
    'final_predictions.csv',
    index=False
)

print("Final predictions saved successfully!")

print("\n===== FINAL PREDICTIONS =====")
display(final_results)

**2. Save Feature Importance**

In [ ]:
# Save feature importance
feature_importance.to_csv(
    'feature_importance.csv',
    index=False
)

print("Feature importance saved successfully!")

print("\n===== FEATURE IMPORTANCE =====")
display(feature_importance)

**3. Save Model Performance**

In [ ]:
# Create performance summary
performance = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'Tuned Random Forest'
    ],
    'MSE': [
        mse_linear,
        mse,
        mse_final
    ],
    'RMSE': [
        np.sqrt(mse_linear),
        np.sqrt(mse),
        rmse
    ],
    'R2 Score': [
        r2_linear,
        r2,
        r2_final
    ]
})

print("\n===== MODEL PERFORMANCE SUMMARY =====")
display(performance)

**Save it:**

In [ ]:
performance.to_csv(
    'model_performance.csv',
    index=False
)

print("Model performance saved successfully!")

display(performance)

**4. Check All Output Files**

In [ ]:
import os

files = [
    'final_predictions.csv',
    'feature_importance.csv',
    'model_performance.csv',
    'final_random_forest_model.pkl',
    'feature_scaler.pkl'
]

status_data = []

for file in files:
    status_data.append({
        "File Name": file,
        "Status": "Available" if os.path.exists(file) else "Not Found"
    })

df_files = pd.DataFrame(status_data)

display(df_files)

**5. Display Final Performance**

In [ ]:
print("\n===== FINAL MODEL SUMMARY =====")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse_final:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2_final:.4f}")

## **Final ML Project Report**

**1. Project Overview**

In [ ]:
project_report = """
# 📘 MACHINE LEARNING PROJECT REPORT

### 🌍 Project
**Global Gini Coefficient Analysis & Forecasting**

### 🤖 Machine Learning Approach

1. **Data Cleaning** (repair of the corrupted Gini column)
2. **Exploratory Data Analysis (EDA)**
3. **Feature Engineering** (lags, rolling means, changes, entity trend)
4. **Time-based Train/Test Split**
5. **Feature Scaling**
6. **Linear Regression & Random Forest Regression**
7. **Hyperparameter Tuning (GridSearchCV + TimeSeriesSplit)**
8. **Model Evaluation (vs. naive baseline)**
9. **Prediction Function**
"""

display(Markdown(project_report))

**2. Dataset Summary**

In [ ]:
dataset_summary = pd.DataFrame({
    "Dataset Information": [
        "Dataset Shape",
        "Number of Features",
        "Number of Target Values"
    ],
    "Value": [
        str(df.shape),
        X.shape[1],
        len(y)
    ]
})

display(Markdown("## 📊 DATASET SUMMARY"))

display(
    dataset_summary.style
    .hide(axis="index")
)

**3. Model Performance**

In [ ]:
# Model Performance
display(Markdown("## 📊 MODEL PERFORMANCE"))

model_performance = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "R² Score": [
        r2_linear,
        r2,
        r2_final
    ]
})

display(
    model_performance.style
    .format({"R² Score": "{:.4f}"})
    .hide(axis="index")
)

# Final Model Metrics
display(Markdown("## 🎯 FINAL MODEL METRICS"))

final_metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R² Score"],
    "Value": [mae, mse_final, rmse, r2_final]
})

display(
    final_metrics.style
    .format({"Value": "{:.4f}"})
    .hide(axis="index")
)

**4. Most Important Features**

In [ ]:
display(Markdown("## ⭐ TOP 10 IMPORTANT FEATURES"))

display(
    feature_importance
    .head(10)
    .style
    .format({"Importance": "{:.4f}"})
    .hide(axis="index")
)

**5. Final Conclusion**

In [ ]:
display(Markdown(f"""
## ✅ CONCLUSION

The Gini column was repaired (digits read as the fractional part of a 0–1 value), after which all values fall in a
valid range (**{df_clean['Gini coefficient'].min():.3f} – {df_clean['Gini coefficient'].max():.3f}**).

On unseen later years, the tuned Random Forest reached **R² = {r2_final:.3f}** (RMSE = {rmse:.4f}); the naive
last-observation baseline reached **R² = {r2_naive:.3f}** (RMSE = {rmse_naive:.4f}).

Gini coefficients are highly persistent over time, so the previous observation is already a strong predictor —
the model should be judged against that baseline, not against zero. Results are most reliable for entities with
long, regular histories; sub-national (urban/rural) series and sparse entities add noise.
"""))

## **Project Conclusion & Portfolio Preparation**

**1. Project Objective**

In [ ]:
project_objective = """
## 🎯 PROJECT OBJECTIVE

The objective of this project is to analyze country-level Gini coefficient data (1963–2025) using Python and to
develop a Machine Learning regression model that predicts an entity's Gini coefficient from its recent history.

The project covers the complete workflow: data repair and preprocessing, exploratory analysis, feature
engineering, model development, time-aware evaluation, hyperparameter tuning, and prediction.
"""

display(Markdown(project_objective))

**2. Technologies Used**

In [ ]:
technologies = [
    "Python", "Pandas", "NumPy", "Matplotlib", "Seaborn", "Plotly",
    "Scikit-learn", "Linear Regression", "Random Forest Regression",
    "GridSearchCV / TimeSeriesSplit", "Joblib"
]

technology_text = "## 🛠️ Technologies & Tools Used\n\n"
for technology in technologies:
    technology_text += f"- ✓ **{technology}**\n"

display(Markdown(technology_text))

**3. Machine Learning Workflow**

In [ ]:
workflow_text = """
## 🔄 Machine Learning Workflow

1. **Data Collection**
2. **Data Cleaning**
3. **Exploratory Data Analysis (EDA)**
4. **Feature Engineering**
5. **Train-Test Split**
6. **Feature Scaling**
7. **Model Training**
8. **Model Evaluation**
9. **Hyperparameter Tuning**
10. **Final Prediction**
11. **Model Saving**
"""

display(Markdown(workflow_text))

**4. Final Results**

In [ ]:
display(Markdown(f"""
## 📊 FINAL MODEL RESULTS

**MAE:** {mae:.4f}

**MSE:** {mse_final:.4f}

**RMSE:** {rmse:.4f}

**R² Score:** {r2_final:.4f}
"""))

**5. Final Project Statement**

In [ ]:
print("""
PROJECT SUMMARY

This project demonstrates an end-to-end Machine Learning workflow on global Gini coefficient data.

A corrupted numeric column was diagnosed and repaired, the data was explored, and lag, rolling and trend
features were engineered per country. Linear Regression and Random Forest models were trained on the
earlier years and tested on the most recent years (a time-based split), and the Random Forest was tuned with
GridSearchCV using time-series cross-validation. Results were compared with a naive 'last value' baseline.

The trained model and feature scaler were saved so they can be reused for future predictions.

Skills demonstrated: Python, data cleaning, data visualization, feature engineering, machine learning,
model evaluation and predictive analytics.
""")

## **𝐄𝐍𝐃**
---